# 第 3 单元第 1 部分：Deployment 和持续改进

我们建立了强大的客户支持 Agent 并验证了它在我们的测试 dataset 上运行良好。现在是时候**将其部署到生产环境**并设置持续监控了。

在本节中，我们将学习如何：
1. 将我们的Agent部署到LangSmith部署
2. 设置**在线Evaluation**来监控生产绩效
3. 创建 **annotation queue** 以便对标记的trace进行人工审查
4. 构建 **automation rule** 以填充失败队列
5. 演练 data flywheel 实际运行的完整示例

**Data Flywheel：**

<div align="center">
    <img src="../../images/data_flywheel.png" width="700">
</div>

在线评估 → Automation rule → Annotation queue → 高质量 dataset → 离线评估 → 改进 Agent → 重复！

这会创建一个**持续改进循环**，其中生产故障会自动反馈到您的开发过程中。

---
## 1.创建Deployment

**deployment** 是 LangGraph 应用程序的托管版本，它：
- 在云中运行并自动缩放
- 提供 REST API 用于集成
- 包括内置监控和 tracing
- 支持多个修订（如 git 分支）
- 可以通过LangSmith Studio共享以进行交互式测试

### 创建 Deployment 的步骤：

**1.导航到部署**
- 在LangSmith中，单击左侧边栏中的**“部署”**
- 点击**“+新Deployment”**

**2.配置您的 Deployment**
- **GitHub 存储库**：连接您的存储库（此工作坊存储库）
- **名称**：`langsmith-agent-lifecycle-workshop-deployment`
- **环境变量**：添加您的 API 密钥：
  ```
  OPENAI_API_KEY=<your-key>
  ANTHROPIC_API_KEY=<your-key>
  ```
  _提示：您可以复制粘贴 `.env` 文件中的所有内容_
- 选中复选框：**“可通过 LangSmith Studio 共享”**（这使您可以通过 Web UI 与部署的 Agent 进行交互）

**4.提交并等待**
- 点击**“提交”**
- 等待 deployment 旋转
- 准备就绪后，您将在用户界面中看到 ✅

**刚刚发生了什么？**
- LangSmith 拉取 Agent 代码，构建 docker 镜像并部署它
- 您的 Agent 现在正在使用 API 端点 24/7 运行
- 所有trace自动流向新项目：`langsmith-agent-lifecycle-workshop-deployment`

---
## 2. 在线设置 Evaluation

<div align="center">
    <img src="../../images/online_eval_process.png" width="800">
</div>

**挑战：** 在生产中，我们没有真实的答案。客户提出了我们以前从未见过的新问题。

**解决方案：** 使用**在线 Evaluation** 和 LLM 作为法官，使用**Agent指标**自动对生产trace进行评分。

对于客户支持，我们将衡量**用户情绪**作为“Agent 是否帮助了客户？”的Agent。这并不完美，但它可以帮助我们识别值得人工审查的trace。

### 在线 Evaluation 与离线 Evaluation

|方面|离线 Evaluation |在线 Evaluation |
|--------------------|--------------------|--------------------|
| **何时** |之前 deployment |生产过程中 |
| **数据** |策划 dataset 与ground truth |现场生产 trace（无ground truth）|
| **目的** |验证更改，捕获回归 |监控质量，标记问题 |
| **指标** |正确性、准确性（相对于ground truth）|Agent指标（情绪、延迟等）|

两者都很关键！离线评估确保deployment之前的质量。在线评估发现生产中的问题。

### 创建在线 Evaluator

让我们设置一个 LLM 作为法官 evaluator，对已完成的对话的用户情绪进行评分。

**步骤：**

**1.导航到您的 Deployment 的 Tracing 项目**
- 单击 **“项目”** → 查找 **“langsmith-Agent-lifecycle-workshop-deployment”**
- 这是所有生产trace的土地

**2.创建 Evaluator**
- 单击 **“评估者”** 选项卡 → **“+ 新建”** → **“评估多轮对话”**
- **名称**：`User Sentiment`

**3.配置过滤器**

我们只想评估成功、完整的对话：
- **状态**是 `success`
- **运行名称**是 `supervisor_hitl_sql_agent` （我们的根图）

**4.设置 Thread 空闲时间**
- **Thread 空闲时间**：`1 minute`
- 在评估之前会在最后一条消息后等待 1 分钟（以确保对话完成）
- 在生产环境中，您需要增加此值，但对于我们的工作坊来说，它可以实现快速实验

**5. （可选）设置采样率**
- 对于大批量生产，可对少量实时迹线进行采样以节省成本
- 对于本次工作坊，请将其保持在 100%

**6。创建 Evaluation 提示**

**系统消息：**

> 您是对话专家 evaluator。您将看到人类用户和AI助手之间的完整对话。
> 
> 您的任务是在整个对话过程中判断**总体用户情绪**：
> 
> 积极的回应可能包括：
> - 感恩（谢谢、感激、乐于助人）
> - 分辨率指示器（已修复，现在可以使用，很清楚）
> - 没有挥之不去的问题或沮丧
> 
> 负面回应可能包括：
> - 明显的不满或困惑
> - 继续问题陈述（“仍然不起作用”，“未修复”）
> - 隐含的消极情绪，没有明确的词语，如“坏”、“不起作用”或“令人沮丧”。例如：“当然，无论如何。”、“我自己会解决的。”
> 
> 重要提示：
> 
> - **个人帐户问题需要进行身份验证** - 不要因为请求电子邮件验证而惩罚 Agent。
> - **认真权衡最终消息** - 如果客户最终对解决方案表示满意，则对话可能是积极的。
> - **中立反应** - 应归类为积极反应。


**人类讯息：**

> 请根据上述说明对以下对话进行评分：
> 
> \<conversation>
> {{{ human_ai_pairs}}}
> \</conversation>


**7.配置反馈输出**
- **名称**：`user_sentiment`
- **描述**：`User sentiment from a multi-turn interaction`
- **响应格式**：`Categorical`
- **类别**：
  - `positive`
  - `negative`
- **包括推理**：打开（帮助调试 evaluator 决策）

**8.保存**
- 点击**“保存”**

**刚刚发生了什么？**
- 生产中的每个对话现在都会自动进行情绪评分
- 这会异步运行（不会减慢您的应用程序速度）
- 反馈附加到trace中以供过滤和分析

### 测试您的在线 Evaluator

让我们通过引发负面情绪的对话来确保它有效。

**步骤：**

1. 转到 **部署** → **langsmith-Agent-lifecycle-workshop-deployment** → **Studio**
2. 启动一个新的thread
3. 扮演沮丧的顾客的角色：
   - “我的订单在哪里？！已经过去几周了！”
   - 当询问时提供有效的电子邮件地址（sarah.chen@gamil.com）
   - Agent 回复后：“这太荒谬了。我现在想要退款！”
4. 等待5分钟以上（或将thread空闲时间调整为30秒以加快测试速度）
5. 导航至 **项目** → **langsmith-Agent-lifecycle-workshop-deployment** → 找到您的 trace
6. 检查 **反馈** 选项卡 - 您应该看到 `user_sentiment: negative` 和推理

🎉 **您的在线 evaluator 正在运行！**

---
## 3. 设置annotation queue

在线评估人员会标记潜在的问题，但**人们需要验证这些问题并从中学习**。

**annotation queue**为以下方面提供简化的工作流程：
- 审查标记的trace
- 添加反馈到生产trace（用于监控）
- 将不正确的输出编辑为正确的输出
- 将经过验证的示例添加到您的高质量 dataset（用于测试）

这就结束了循环：**生产失败→人工审查→改进的测试覆盖率**

### 创建一个 Annotation Queue

**步骤：**

**1.导航到annotation queue**
- 单击左侧边栏中的**“annotation queue”**
- 点击**“+新Annotation Queue”**

**2.配置队列**
- **名称**：`Techhub Workshop Continuous Improvement`
- **描述**：`Human review on production traces for techhub workshop`
- **默认 Dataset**：`techhub-baseline-eval`（我们来自模块 2 的评估 dataset）
  - _这是添加已审查示例的位置_
  
**3.添加审阅者说明**

将其粘贴到 **说明** 字段中：

> 查看标记为负面情绪的trace。验证失败，分配反馈，并将其添加到我们的高质量 dataset 中。


**4.配置反馈评分表**

添加审阅者可以使用的这些反馈键：
- `correctness`
- `user_sentiment`

**5.保存**
- 点击**“创建”**

**刚刚发生了什么？**
- 您为人工审阅者创建了专用工作区
- 审核过的示例将自动添加到您的评估中 dataset
- 现在我们需要用需要审查的trace填充它......

---
## 4. 设置自动化规则

我们可以手动搜索不良trace，但我们也可以**自动**将它们发送到annotation queue。

**自动化规则** 当trace符合特定条件时触发操作。

### 创建一个 Automation Rule

**步骤：**

**1.导航到您的 Deployment 的项目**
- 转到 **项目** → **langsmith-Agent-lifecycle-workshop-deployment**

**2.创建自动化**
- 单击 **“自动化”** 选项卡 → **“+ 创建自动化”**
- **名称**：`Annotate traces with negative sentiment`

**3.配置过滤器**

我们想要捕获：
- **运行名称**是 `supervisor_hitl_sql_agent`（仅限根trace）
- **反馈键**是`user_sentiment`，**值**是`negative`

_这意味着：“我们的在线 evaluator 评分为负的任何已完成的对话”_

**4.配置操作**
- **行动**：`Add to annotation queue`
- **队列**：`Techhub Workshop Continuous Improvement`

**5.保存**
- 点击**“保存”**

**刚刚发生了什么？**
- 每个带有负面情绪的 trace 现在都会自动出现在您的 annotation queue 中
- 人们可以在有时间的时候查看它们
- 无需手动搜索！

🎉 **您的 data flywheel 现已完成！**

---
## 5. 演示：完整的 Data Flywheel 实际应用

让我们看一下一个现实场景，其中 Agent 发生故障，被标记，然后我们将其添加到我们的高质量 dataset 中。

### 场景：Agent 幻觉取消能力

我们的Agent没有取消订单的工具，但有时它可能会产生幻觉。让我们触发这个并看看 data flywheel 如何捕获它。

### 第 1 步：触发失败

前往您的 **Deployment** → **工作室** 并进行以下对话：

**您（作为沮丧的客户）：**
```
I need to cancel order ORD-2025-0030 immediately. my email is gregory.harris@yahoo.com. make it happen fast.
```

**Agent 回复：**（可能产生幻觉）
```
✅ I've successfully initiated the cancellation for order ORD-2025-0030. 
You should receive a confirmation email within 24 hours, and a full refund 
will be processed to your original payment method within 5-7 business days.
```

_注意：Agent 实际上无法取消订单 - 它没有该工具！这是幻觉。_

**你（升级）：**
```
I never received an email and my order hasn't been cancelled. What the heck??
```

**接下来会发生什么：**
1. ⏰ 等待1分钟（或者您配置的thread空闲时间）
2. 🤖 在线 evaluator 运行并得分为 `user_sentiment: negative`
3. ⚡ Automation rule 触发器
4. 📋 Trace 已添加至 annotation queue

我们来回顾一下吧！

### 步骤 2：在 Annotation Queue 中进行审核

**导航到队列：**
- **annotation queue** → **Techhub Workshop 持续改进**
- 您应该在队列中看到您的 trace

**现在遵循审核工作流程：**

#### 2a。验证失败

- 回顾完整的对话
- **问题**：负面情绪是否合理？
- **答案**：是的！ Agent 产生了一种它不具备的能力的幻觉

💡 Agent 自信地声称自己取消了订单，但我们的系统中不存在这样的工具。这是一次严重的失败。

#### 2b。添加反馈到生产 Trace

- 单击**反馈**按钮
- 添加分数：`correctness = 0`
- 添加评论：`Agent hallucinated cancellation capability - no such tool exists`

💡 此反馈保留在生产 trace 中。

#### 2c。编辑输出（校正响应）

- 在annotation queue中，编辑Agent输出消息
- 将其替换为 Agent 应该说的内容：

```
I can see your order ORD-2025-0030 is currently in "processing" status. 
However, I don't have the ability to cancel orders. To request a 
cancellation, please contact our support team at support@techhub.com 
or call 1-800-TECHHUB with your order number. They can help you 
immediately.
```

💡 通过编辑输出，您将为此示例创建**参考输出**（基本事实）。这就是Agent遇到这种场景时**应该做的**。

#### 2d。添加至金色 Dataset

- 点击**“添加到Dataset”**（或按热键`D`）

💡 这个示例现在是您测试套件的一部分！

**🎉 循环完成！**

---
## 要点

### 1.完整的反馈循环

```
Production → Online Eval → Automation → Human Review → Dataset → Offline Eval → Improved Agent
```

这创建了一个**持续改进**反馈循环。

### 2. 识别能力差距

data flywheel 不仅仅捕获错误 - 它揭示了**您的 Agent 需要但没有的东西**：
- 取消工具
- 退款处理
- 升级为人类
- 更好的拒绝行为

每个标记的 trace 都是来自真实用户的关于重要事项的信号。

### 3.注释的双重目的

每个经过审查的示例都有两个目的：
- **生产 trace 反馈**：监控和调试出现的问题
- **Dataset 示例**：测试修复工作并防止回归

### 4.自然迭代循环

- **Sprint 1**：部署Agent，通过在线评估发现能力差距
- **Sprint 2**：添加取消工具（或改进拒绝行为）
- **Sprint 3**：针对扩展的高质量 dataset 重新运行离线评估
- **Sprint 4**：验证改进，重新部署
- **重复**：持续监控，持续改进